# Исследование игр
- Автор: Шалак Лидия
- Дата: 7 сентября 2026 г

### Цели и задачи проекта
В этой тетрадке используем датасет `https://code.s3.yandex.net/datasets/new_games.csv`, который содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр. Проведём предобработку данных: определим пропущенные и ошибочные значения, правильные названия столбцов и типов данных, найдём дубликаты. Отбёрем данные за период, на которых проведём категоризацию по оценкам игроков и оценкам критиков и выведем топ 7 платформ по количеству игр.

### Описание данных
В проекте будут использованы данные датасета `https://code.s3.yandex.net/datasets/new_games.csv` с таким описанием:

- `Name` — название игры;
- `Platform` — название платформы;
- `Year of Release` — год выпуска игры;
- `Genre` — жанр игры;
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий);
- `EU sales` — продажи в Европе (в миллионах проданных копий);
- `JP sales` — продажи в Японии (в миллионах проданных копий);
- `Other sales` — продажи в других странах (в миллионах проданных копий);
- `Critic Score` — оценка критиков (от 0 до 100);
- `User Score` — оценка пользователей (от 0 до 10);
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию;

In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем данные из датасета hotel_dataset.csv в датафрейм hotels
new_games = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

In [3]:
# Выводим информацию о датафрейме
new_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выводим первые строки датафрейма на экран
new_games.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет `/datasets/new_games.csv` содержит 11 столбцов и 16956 строк, в которых представлена информация об играх, их продажах, оценках и рейтинге. 

Изучим типы данных и их корректность:

- **Числовые значения с плавающей запятой (float64).** 4 столбца: `Year of Release` – год выхода представлен типом `float64`. `NA sales`,` Other sales` – продажи могут содержать дробные значения. `Critic Score` – верное решение.
- **Строковые данные (object).** 7 столбцов:
    - `Name`, `Platform` и `Genre` содержат строковую информацию (название игры, платформы, жанр), что логично для текстовых данных. Здесь тип данных `object` подходит.
    - `EU sales`, `JP sales`, `User Score` представляют дробные значения. Для таких данных рекомендуется использовать тип `float64`, чтобы привести к единому типу со столбцами `NA sales`,` Other sales` и `Critic Score`.
    - `Rating ` хранит текстовые данные, но их можно рассматривать как категориальные признаки. В этом случае можно использовать тип `category`, чтобы улучшить производительность и оптимизировать память, если набор значений ограничен и заведомо известен.

После анализа типов данных видно, что большинство столбцов корректно представлены. Однако для оптимизации можно использовать целочисленные типы с уменьшенной разрядностью и преобразовать строковые данные  в тип `float64`.

In [5]:
# Выводим названия всех столбцов датафрейма
print(new_games.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


Приведём названия столбцов к стилю snake_case: заменим на нижний регистр, заменим пробелы на подчёркивания.

In [6]:
# Приводим названия столбцов к стилю snake_case: нижний регистр и подчёркивания вместо пробелов
new_games.columns = new_games.columns.str.lower().str.replace(' ', '_')
print(new_games.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


In [7]:
# Преобразуем eu_sales, jp_sales, user_score в тип float64
cols_to_float = ['eu_sales', 'jp_sales', 'user_score']
new_games[cols_to_float] = new_games[cols_to_float].apply(pd.to_numeric, errors='coerce')

# Понижаем разрядность всех столбцов с типом float64
float_cols = new_games.select_dtypes(include='float64').columns
new_games[float_cols] = new_games[float_cols].apply(pd.to_numeric, downcast='float')

# Преобразуем rating в тип category
new_games['rating'] = new_games['rating'].astype('category')

new_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   name             16954 non-null  object  
 1   platform         16956 non-null  object  
 2   year_of_release  16681 non-null  float32 
 3   genre            16954 non-null  object  
 4   na_sales         16956 non-null  float32 
 5   eu_sales         16950 non-null  float32 
 6   jp_sales         16952 non-null  float32 
 7   other_sales      16956 non-null  float32 
 8   critic_score     8242 non-null   float32 
 9   user_score       7688 non-null   float32 
 10  rating           10085 non-null  category
dtypes: category(1), float32(7), object(3)
memory usage: 878.1+ KB


Столбцы `eu_sales`, `jp_sales`, `user_score` приведены к float64. Все столбцы типа float64 понижены до float32. Нечисловые значения (например, unknown) в `user_score` стали NaN. Преобразовали `rating` к типу `category`. Память датафрейма с 1.4+ MB упала до 878.1+ KB.

In [27]:
# Выводим количество пропущенных строк в датафрейме
new_games.isna().sum()

name                  0
platform              0
year_of_release     269
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8576
user_score         9123
rating                0
dtype: int64

In [9]:
# Подсчитываем процент строк с пропусками
new_games.isna().sum() / len(new_games) * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

In [10]:
# Проверяем, совпадают ли пропуски в name и genre
print('Пропуски в name и genre в одних и тех же строках:', (new_games['name'].isna() == new_games['genre'].isna()).all())
print(new_games[new_games['name'].isna() | new_games['genre'].isna()])
print()
print('Уникальные значения rating:', new_games['rating'].unique())
print()
print('Медианный год выхода среди игр без critic_score:', new_games.loc[new_games['critic_score'].isna(), 'year_of_release'].median())
print('Медианный год выхода среди игр с critic_score:', new_games.loc[new_games['critic_score'].notna(), 'year_of_release'].median())

Пропуски в name и genre в одних и тех же строках: True
      name platform  year_of_release genre  na_sales  eu_sales  jp_sales  \
661    NaN      GEN           1993.0   NaN      1.78      0.53      0.00   
14439  NaN      GEN           1993.0   NaN      0.00      0.00      0.03   

       other_sales  critic_score  user_score rating  
661           0.08           NaN         NaN    NaN  
14439         0.00           NaN         NaN    NaN  

Уникальные значения rating: ['E', NaN, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Categories (8, object): ['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T']

Медианный год выхода среди игр без critic_score: 2008.0
Медианный год выхода среди игр с critic_score: 2007.0


In [11]:
# Проверяем гипотезу: пропуски в critic_score/user_score характерны для игр, продававшихся в Японии
jp_only = (new_games['na_sales'] + new_games['eu_sales'] + new_games['other_sales']) < 0.01
print('Доля JP-игр среди всех строк:', jp_only.mean())
print('Доля JP-игр среди строк без critic_score:', jp_only[new_games['critic_score'].isna()].mean())
print()
print('Доля пропусков rating по платформам (топ-10):')
print(new_games.groupby('platform')['rating'].apply(lambda s: s.isna().mean()).sort_values(ascending=False).head(10))

Доля JP-игр среди всех строк: 0.18901863647086578
Доля JP-игр среди строк без critic_score: 0.35724122102364014

Доля пропусков rating по платформам (топ-10):
platform
2600    1.0
PCFX    1.0
WS      1.0
TG16    1.0
SNES    1.0
GB      1.0
SCD     1.0
SAT     1.0
GEN     1.0
GG      1.0
Name: rating, dtype: float64


### Промежуточный вывод по пропущенным значениям

Пропуски есть в 8 из 11 столбцов:

**name** — 2 пропуска (0.01%). **genre** — 2 пропуска (0.01%). **year_of_release** — 275 пропусков (1.62%). **eu_sales** — 6 пропусков (0.04%). **jp_sales** — 4 пропуска (0.02%). **rating** — 6871 пропуск (40.52%). **critic_score** — 8714 пропусков (51.39%). **user_score** — 9268 пропусков (54.66%).

Возможные причины пропусков.

`name` и `genre` пропущены в одних и тех же 2 строках (обе — платформа GEN, 1993 год). Это единичная несущественная ошибка.

`eu_sales` и `jp_sales` проспуски в виде строки `unknown` вместо числового значения — то есть продажи в этих регионах либо не отслеживались, либо не были указаны источником.

`year_of_release` может отсутствовать для старых изданий, по которым источник данных не сохранил точную дату выхода.

`rating` пропущен в 100% строк для старых платформ — 2600, GB, SNES, GEN, SAT, GG, TG16, WS, SCD, PCFX. Система ESRB появилась  в 1994 году, поэтому у игр на более старых платформах рейтинг не указывали.

`critic_score` и `user_score` пропущены почти в половине строк. Проверка гипотезы показала: доля игр, которые продавались в Японии (na_sales + eu_sales + other_sales ≈ 0), среди строк без critic_score — 35.7%, тогда как в среднем по датасету таких игр только 18.9%. То есть игры западного релиза реже получали оценки критиков и пользователей.

Что можно сделать:

`name`, `genre`: строки единичные и не несут значимой информации без названия — удалить.

`year_of_release`: пропусков немного (1.6%), но достоверно восстановить год без внешних данных нельзя — оставить как есть и учитывать пропуски при анализе по годам.

`eu_sales`, `jp_sales`: заменить на среднее по платформе и году выхода игры.

`rating`: пропуск информативен сам по себе (игра не имеет рейтинга ESRB) — заменить на индикатор-заглушку, предварительно убедившись, что такого значения нет среди реальных категорий.

`critic_score`, `user_score`: пропусков слишком много (>50%), а замена на среднее/медиану исказила бы распределение оценок — оставить как есть и учитывать пропуски как «оценка отсутствует», а не как ноль.

In [12]:
# Удаляем строки с пропущенными name и genre: их всего 2, и без названия игра не несёт полезной информации
new_games = new_games.dropna(subset=['name', 'genre']).reset_index(drop=True)

new_games.isna().sum()

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8712
user_score         9266
rating             6869
dtype: int64

In [13]:
# Заполняем пропуски в eu_sales и jp_sales средним по платформе и году выхода игры
for column in ['eu_sales', 'jp_sales']:
    group_mean = new_games.groupby(['platform', 'year_of_release'])[column].transform('mean')
    new_games[column] = new_games[column].fillna(group_mean)
    platform_mean = new_games.groupby('platform')[column].transform('mean')
    new_games[column] = new_games[column].fillna(platform_mean)

In [14]:
# Проверяем, что пропуски в eu_sales и jp_sales устранены
new_games[['eu_sales', 'jp_sales']].isna().sum()

eu_sales    0
jp_sales    0
dtype: int64

In [15]:
# Проверяем, что unknown не используется как реальное значение rating, и заполняем пропуски этим индикатором
print('unknown уже используется в rating:', 'unknown' in new_games['rating'].cat.categories)
new_games['rating'] = new_games['rating'].cat.add_categories('unknown')
new_games['rating'] = new_games['rating'].fillna('unknown')

new_games['rating'].value_counts()

unknown уже используется в rating: False


rating
unknown    6869
E          4037
T          3005
M          1587
E10+       1441
EC            8
K-A           3
RP            3
AO            1
Name: count, dtype: int64

In [16]:
# Итоговая проверка пропусков после обработки
new_games.isna().sum()

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8712
user_score         9266
rating                0
dtype: int64

### Итог обработки пропусков

Удалили 2 строки с пропущенными `name` и `genre`, т.к. их доля несущественна. Заполнили пропуски в `eu_sales` и `jp_sales` средним по платформе и году выхода игры (с запасным вариантом — средним по платформе, если группа платформа+год состояла из одних пропусков). Пропуски в `rating` заменили на индикатор-заглушку `unknown`. Пропуски в `year_of_release`, `critic_score` и `user_score` оставили как есть: их доля либо мала и невосстановима без внешних источников (`year_of_release`), либо слишком велика для надёжной замены на среднее/медиану без искажения распределения (`critic_score`, `user_score`).

## Явные и неявные дубликаты в данных

Изучим уникальные значения в категориальных столбцах (`genre`, `platform`, `rating`, `year_of_release`) на предмет неявных дубликатов — например, одинаковых по смыслу значений, записанных в разном регистре или с опечатками. При необходимости нормализуем текстовые данные и после этого проверим датафрейм на явные дубликаты строк.

In [17]:
# Изучаем уникальные значения в категориальных столбцах
print('Уникальные жанры:', sorted(new_games['genre'].dropna().unique()))
print()
print('Уникальные платформы:', sorted(new_games['platform'].unique()))
print()
print('Уникальные рейтинги:', new_games['rating'].cat.categories.tolist())
print()
print('Уникальные годы выпуска:', sorted(new_games['year_of_release'].dropna().unique()))

Уникальные жанры: ['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', 'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform', 'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing', 'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']

Уникальные платформы: ['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']

Уникальные рейтинги: ['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'unknown']

Уникальные годы выпуска: [np.float32(1980.0), np.float32(1981.0), np.float32(1982.0), np.float32(1983.0), np.float32(1984.0), np.float32(1985.0), np.float32(1986.0), np.float32(1987.0), np.float32(1988.0), np.float32(1989.0), np.float32(1990.0), np.float32(1991.0), np.float32(1992.0), np.float32(1993.0), np.float32(1994.0), np.float32(1995.0), np.float32(1996.0), np.float32(1997.0)

In [18]:
# Оцениваем масштаб неявных дубликатов: сравниваем число уникальных значений до и после приведения к единому регистру
print('genre: сейчас', new_games['genre'].nunique(), 'значений, после нижнего регистра —', new_games['genre'].str.lower().nunique())
print('platform: сейчас', new_games['platform'].nunique(), 'значений, после нижнего регистра —', new_games['platform'].str.lower().nunique())
print('rating: сейчас', new_games['rating'].nunique(), 'значений, после верхнего регистра —', new_games['rating'].astype(str).str.upper().nunique())

genre: сейчас 24 значений, после нижнего регистра — 12
platform: сейчас 31 значений, после нижнего регистра — 31
rating: сейчас 9 значений, после верхнего регистра — 9


In [19]:
# Нормализуем текстовые данные: name и genre приводим к нижнему регистру, rating — к верхнему
new_games['name'] = new_games['name'].str.lower()
new_games['genre'] = new_games['genre'].str.lower()
new_games['rating'] = new_games['rating'].astype(str).str.upper().astype('category')

print('Уникальных жанров после нормализации:', new_games['genre'].nunique())
print('Уникальные рейтинги после нормализации:', new_games['rating'].cat.categories.tolist())

Уникальных жанров после нормализации: 12
Уникальные рейтинги после нормализации: ['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']


In [20]:
# Проверяем явные дубликаты строк после нормализации текстовых данных
print('Явных дубликатов:', new_games.duplicated().sum())
new_games[new_games.duplicated(keep=False)].sort_values(by=['name', 'platform', 'year_of_release'])

Явных дубликатов: 241


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
15189,beyblade burst,3DS,2016.0,role-playing,0.00,0.00,0.03,0.00,NaN,NaN,UNKNOWN
15190,beyblade burst,3DS,2016.0,role-playing,0.00,0.00,0.03,0.00,NaN,NaN,UNKNOWN
15299,11eyes: crossover,X360,2009.0,adventure,0.00,0.00,0.02,0.00,NaN,NaN,UNKNOWN
15300,11eyes: crossover,X360,2009.0,adventure,0.00,0.00,0.02,0.00,NaN,NaN,UNKNOWN
4859,18 wheeler: american pro trucker,PS2,2001.0,racing,0.20,0.15,0.00,0.05,61.0,5.7,E
...,...,...,...,...,...,...,...,...,...,...,...
2908,yu-gi-oh! the falsebound kingdom,GC,2002.0,strategy,0.49,0.13,0.07,0.02,NaN,NaN,UNKNOWN
6694,zoo resort 3d,3DS,2011.0,simulation,0.11,0.09,0.03,0.02,NaN,NaN,E
6695,zoo resort 3d,3DS,2011.0,simulation,0.11,0.09,0.03,0.02,NaN,NaN,E
8155,zumba fitness rush,X360,2012.0,sports,0.00,0.16,0.00,0.02,73.0,6.2,E10+


In [21]:
# Удаляем явные дубликаты и сбрасываем индекс
new_games = new_games.drop_duplicates().reset_index(drop=True)
print('Строк после удаления дубликатов:', len(new_games))
print('Осталось дубликатов:', new_games.duplicated().sum())

Строк после удаления дубликатов: 16713
Осталось дубликатов: 0


### Промежуточный вывод по дубликатам

Среди категориальных столбцов неявные дубликаты нашлись только в `genre`: одни и те же жанры были записаны и заглавными буквами (`ACTION`), и с заглавной первой буквой (`Action`) — из-за этого вместо 12 реальных жанров в данных числилось 24 уникальных значения. В `platform` и `rating` неявных дубликатов не оказалось — количество уникальных значений не изменилось после приведения к единому регистру.

Нормализовали текстовые данные: `name` и `genre` привели к нижнему регистру, `rating` — к верхнему (включая индикатор-заглушку `unknown`, который стал `UNKNOWN`).

После нормализации в данных обнаружилось 241 явное дублирование строк (482 строки участвовали в дублях) — это записи, которые совпадали по всем столбцам, но раньше не считались дубликатами из-за разного регистра в `genre`. Удалили повторы функцией `drop_duplicates()` и сбросили индекс: строк в датафрейме стало 16 713 вместо 16 954.

## Фильтрация данных

Отбираем данные за период с 2000 по 2013 год включительно и сохраняем срез в отдельном датафрейме `df_actual`.

In [22]:
# Отбираем данные за период с 2000 по 2013 год включительно
df_actual = new_games[(new_games['year_of_release'] >= 2000) & (new_games['year_of_release'] <= 2013)].reset_index(drop=True)
print('Строк в исходном датафрейме:', len(new_games))
print('Строк в срезе df_actual:', len(df_actual))
df_actual['year_of_release'].describe()

Строк в исходном датафрейме: 16713
Строк в срезе df_actual: 12781


count    12781.000000
mean      2007.106445
std          3.444318
min       2000.000000
25%       2004.000000
50%       2008.000000
75%       2010.000000
max       2013.000000
Name: year_of_release, dtype: float64

## Категоризация данных

Разбиваем оценки пользователей `user_score` на категории: `высокая оценка` — от 8 до 10 включительно, `средняя оценка` — от 3 до 8, не включая правую границу, `низкая оценка` — от 0 до 3, не включая правую границу. Разбиваем оценки критиков `critic_score`: `высокая оценка` — от 80 до 100 включительно, `средняя оценка` — от 30 до 80, не включая правую границу, `низкая оценка` — от 0 до 30, не включая правую границу. Строки с пропущенной исходной оценкой выделяем в отдельную категорию `нет оценки`.

In [23]:
# Категоризация по оценке пользователей через функцию с явной обработкой пропусков
def user_score_category(score):
    if pd.isna(score):
        return 'нет оценки'
    elif score >= 8:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

df_actual['user_score_category'] = df_actual['user_score'].apply(user_score_category)
df_actual['user_score_category'].value_counts(dropna=False)

user_score_category
нет оценки        6298
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: count, dtype: int64

In [24]:
# Категоризация по оценке критиков через функцию с явной обработкой пропусков
def critic_score_category(score):
    if pd.isna(score):
        return 'нет оценки'
    elif score >= 80:
        return 'высокая оценка'
    elif score >= 30:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

df_actual['critic_score_category'] = df_actual['critic_score'].apply(critic_score_category)
df_actual['critic_score_category'].value_counts(dropna=False)

critic_score_category
нет оценки        5612
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: count, dtype: int64

## Топ-7 платформ по количеству игр

Выделяем 7 платформ с наибольшим числом выпущенных игр за весь период 2000–2013.

In [25]:
# Топ-7 платформ по количеству выпущенных игр за 2000-2013 годы
top_platforms = df_actual['platform'].value_counts().head(7)
top_platforms

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

## Основной вывод

Из очищенного датафрейма `new_games` нашли и удалили 241 явный дубликат строк, возникший из-за разного регистра в `genre` (`ACTION` и `Action` считались разными жанрами, пока не нормализовали текстовые данные) — после этого в `new_games` осталось 16 713 строк вместо 16 954.

Из обновлённого `new_games` выделили срез `df_actual` — только игры с годом выхода от 2000 до 2013 включительно (12 781 строка из 16 713, то есть около 76.5% всех данных). Этот период охватывает расцвет консолей шестого и седьмого поколений (PS2, DS, Wii, PSP, X360, PS3) и достаточно длинный, чтобы анализировать динамику продаж, жанров и платформ без искажений.

В `df_actual` добавили два новых категориальных поля. `user_score_category` делит игры по пользовательской оценке (шкала 0–10) на `низкая оценка` (0–3), `средняя оценка` (3–8) и `высокая оценка` (8–10); строки без исходной оценки получают отдельную категорию `нет оценки`, а не относятся условно к одной из содержательных групп. `critic_score_category` устроено так же, но по оценке критиков (шкала 0–100) с границами 0–30, 30–80 и 80–100. Категория `нет оценки` по-прежнему многочисленна (`user_score_category`: 6298 из 12 781, `critic_score_category`: 5612 из 12 781) — это ожидаемо, ведь исходные `user_score` и `critic_score` были пропущены больше чем в половине строк ещё на этапе обработки пропусков.

Определили топ-7 платформ по числу выпущенных игр за 2000–2013 годы: PS2 (2127), DS (2120), Wii (1275), PSP (1180), X360 (1121), PS3 (1087) и GBA (811).